In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import torch.nn as nn
import pennylane as qml
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [2]:
# Data preparation
X, y = make_moons(n_samples=200, noise=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train), X_train.shape}, Testing samples: {len(X_test), X_test.shape}")

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.from_numpy(X_train).float()
y_train = torch.from_numpy(y_train).float()
X_test = torch.from_numpy(X_test).float()
y_test = torch.from_numpy(y_test).float()

Training samples: (160, (160, 2)), Testing samples: (40, (40, 2))


In [28]:
from models.hybrid_nn import HybridBinaryModel
from models.qcircuits.vqc import quantum_net

binary_model = HybridBinaryModel(n_qubits=4, n_layers=3, qcircuit=quantum_net)
optimizer = torch.optim.Adam(binary_model.parameters(), lr=0.0005)
bce_loss = nn.BCEWithLogitsLoss() # Binary Cross-Entropy Loss with Logits

In [29]:
from models.methods import training
device = torch.device("cpu")

binary_model_history = training(binary_model, X_train, y_train, bce_loss, optimizer, device, epochs=350, visualize=True, refresh_rate=5)

Training Progress: 100%|██████████| 350/350 [51:01<00:00,  8.75s/it, train_acc=0.759, train_loss=0.611]  


In [41]:
from utils import create_gif

create_gif(image_dir='./figures/progress_steps/decision_boundary', output_path='./figures/decision_boundary_training.gif', duration_ms=200)
create_gif(image_dir='./figures/progress_steps/training_curves', output_path='./figures/training_curves.gif', duration_ms=200, image_pattern='training_curves')

Creating GIF from 70 images in ./figures/progress_steps/decision_boundary...
Saving GIF to ./figures/decision_boundary_training.gif with frame duration 200 ms.
Creating GIF from 70 images in ./figures/progress_steps/training_curves...
Saving GIF to ./figures/training_curves.gif with frame duration 200 ms.
